# Лабораторная работа №4 по ТМО
## [Ваше ФИО]
## [Ваша группа]

#### Цель работы. Изучение линейных моделей, SVM и деревьев решений.

## 1. Импорт библиотек и загрузка датасета

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, f1_score

# Загрузка датасета Wine
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')
feature_names = wine.feature_names

print("Датасет: Wine")
print(f"Классы: {wine.target_names}")
print(f"Задача: классификация вин по химическому составу")

## 2. Проверка наличия пропусков и необходимости предобработки

In [ ]:
print(f"Количество образцов: {X.shape[0]}")
print(f"Количество признаков: {X.shape[1]}")
print(f"Признаки: {list(feature_names)}")
print(f"Диапазон целевой переменной: [{y.min()}, {y.max()}]")
print(f"\nПропуски в признаках:")
print(X.isnull().sum())
print(f"Пропуски в целевой переменной: {y.isnull().sum()}")

#### В наборе находятся только числовые признаки, выявлено также отсутствие пропусков, так что дальнейшая обработка данных не требуется.

## 3. Разделение выборки на обучающую и тестовую

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} образцов ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Тестовая выборка: {X_test.shape[0]} образцов ({X_test.shape[0]/len(X)*100:.0f}%)")

## 4. Обучение моделей

### 4.1 Логистическая регрессия

In [ ]:
# Логистическая регрессия (внутри Pipeline со стандартизацией)
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("Логистическая регрессия обучена.")

### 4.2 SVM (метод опорных векторов)

In [ ]:
# SVM (внутри Pipeline со стандартизацией)
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, random_state=42))
])
svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)

print("SVM обучена.")

### 4.3 Дерево решений

In [ ]:
# Дерево решений
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)

print("Дерево решений обучено.")

## 5. Оценка качества моделей

Метрики: **Accuracy** (доля правильных ответов) и **F1-macro** (среднее F1 по всем классам)

In [ ]:
results = {
    'Логистическая регрессия': (y_pred_lr,),
    'SVM': (y_pred_svm,),
    'Дерево решений': (y_pred_tree,),
}

print(f"{'Модель':<28} {'Accuracy':>10} {'F1-macro':>10}")
print('-' * 52)
for name, (y_pred,) in results.items():
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    print(f"{name:<28} {acc:>10.4f} {f1:>10.4f}")

## 6. Важность признаков дерева решений

In [ ]:
importances = tree_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 5))
plt.bar(range(X.shape[1]), importances[indices], color='steelblue')
plt.xticks(range(X.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.title('Важность признаков в дереве решений')
plt.ylabel('Важность')
plt.tight_layout()
plt.show()

## 7. Визуализация дерева решений

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    tree_model,
    feature_names=list(X.columns),
    class_names=wine.target_names,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Визуализация дерева решений')
plt.show()

## 8. Правила дерева решений в текстовом виде

In [ ]:
tree_rules = export_text(tree_model, feature_names=list(X.columns))
print("Правила дерева решений:")
print(tree_rules)

## Вывод

#### В ходе работы были обучены три модели классификации на датасете Wine. SVM показала наилучшее качество по метрикам Accuracy и F1-macro благодаря эффективному разделению признаков в многомерном пространстве. Логистическая регрессия показала схожие результаты. Дерево решений позволило наглядно интерпретировать процесс принятия решений и определить наиболее важные химические характеристики вина. Ключевым признаком оказался flavanoids.